In [1]:
!pip install transformers bitsandbytes accelerate peft trl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/derma_project'
DATA_DIR = f'{PROJECT_DIR}/data'
JSON_PATH = f'{DATA_DIR}/generated_reports_clean.json'

Mounted at /content/drive


In [3]:
import json
import torch
from torch.utils.data import Dataset
from PIL import Image

with open(JSON_PATH, 'r') as f:
    reports = json.load(f)

print(f"Loaded {len(reports)} training examples")

Loaded 2376 training examples


In [18]:
class DermatologyDataset(Dataset):
    def __init__(self, reports, processor):
        self.reports = reports
        self.processor = processor

    def __len__(self):
        return len(self.reports)

    @property
    def column_names(self):
        return None

    def __iter__(self):
        for i in range(len(self)):
            yield self[i]

    def __getitem__(self, idx):
        item = self.reports[idx]

        image = Image.open(item['image_path']).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": f"""You are a dermatology assistant analyzing a dermatoscopic image.

Patient information:
- Age: {item['age']} years old
- Sex: {item['sex']}
- Lesion location: {item['localization']}

Analyze the visual features of this lesion and produce a structured clinical report."""}
                ]
            },
            {
                "role": "assistant",
                "content": item['report']
            }
        ]

        input_text = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=False
        )

        inputs = self.processor(
            image,
            input_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        )

        labels = inputs['input_ids'].clone()
        input_ids = inputs['input_ids'][0]
        assistant_start = self._find_assistant_start(input_ids)
        labels[0, :assistant_start] = -100
        inputs['labels'] = labels

        return {k: v.squeeze(0) for k, v in inputs.items()}

    def _find_assistant_start(self, input_ids):
        tokens = input_ids.tolist()
        for i in range(len(tokens) - 1, -1, -1):
            if tokens[i] == 78191:
                return i + 1
        return 0

In [5]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoProcessor

login(token=userdata.get('HF_TOKEN'))

processor = AutoProcessor.from_pretrained(
    "meta-llama/Llama-3.2-11B-Vision-Instruct"
)

print("Processor loaded")

preprocessor_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `MllamaImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Processor loaded


In [6]:
dataset = DermatologyDataset(reports, processor)

print(f"Dataset size: {len(dataset)}")

example = dataset[0]

print(f"\nKeys in example: {list(example.keys())}")
print(f"\nShapes:")
for key, value in example.items():
    print(f"  {key}: {value.shape}")

Dataset size: 2376

Keys in example: ['input_ids', 'attention_mask', 'pixel_values', 'aspect_ratio_ids', 'aspect_ratio_mask', 'cross_attention_mask', 'labels']

Shapes:
  input_ids: torch.Size([292])
  attention_mask: torch.Size([292])
  pixel_values: torch.Size([1, 4, 3, 560, 560])
  aspect_ratio_ids: torch.Size([1])
  aspect_ratio_mask: torch.Size([1, 4])
  cross_attention_mask: torch.Size([292, 1, 4])
  labels: torch.Size([292])


In [7]:
labels = example['labels']
input_ids = example['input_ids']

total_tokens = len(labels)
masked_tokens = (labels == -100).sum().item()
unmasked_tokens = (labels != -100).sum().item()

print(f"Total tokens: {total_tokens}")
print(f"Masked (user message): {masked_tokens}")
print(f"Unmasked (assistant response): {unmasked_tokens}")
print(f"\nLoss will be computed on {unmasked_tokens} tokens")

Total tokens: 292
Masked (user message): 60
Unmasked (assistant response): 232

Loss will be computed on 232 tokens


In [8]:
assistant_token_ids = input_ids[labels != -100]
decoded = processor.decode(assistant_token_ids, skip_special_tokens=True)

print("Assistant tokens decode to:")
print(decoded[:300])


Assistant tokens decode to:


DERMATOSCOPIC ANALYSIS REPORT

Patient: 80-year-old male, lesion located on back.

Visual features observed:
- A prominent, delicate, and regular brown pigment network is visible at the periphery of the lesion.
- The central and superior, slightly elevated component displays darker brown, relative


In [9]:
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import torch

print("Imports successful")

Imports successful


In [10]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print("LoRA config created")
print(f"Rank: {lora_config.r}")
print(f"Alpha: {lora_config.lora_alpha}")
print(f"Scaling factor: {lora_config.lora_alpha / lora_config.r}")
print(f"Target modules: {lora_config.target_modules}")

LoRA config created
Rank: 8
Alpha: 16
Scaling factor: 2.0
Target modules: {'v_proj', 'o_proj', 'k_proj', 'q_proj'}


In [11]:
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import get_peft_model, prepare_model_for_kbit_training
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(model_id)
print("Model loaded")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded


In [12]:
model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, lora_config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")

Total parameters:     5,910,996,515
Trainable parameters: 11,796,480
Trainable percentage: 0.1996%


In [13]:
for name, module in list(model.named_modules())[:50]:
    if 'lora' in name.lower():
        print(name)

base_model.model.model.vision_model.transformer.layers.0.self_attn.q_proj.lora_dropout
base_model.model.model.vision_model.transformer.layers.0.self_attn.q_proj.lora_dropout.default
base_model.model.model.vision_model.transformer.layers.0.self_attn.q_proj.lora_A
base_model.model.model.vision_model.transformer.layers.0.self_attn.q_proj.lora_A.default
base_model.model.model.vision_model.transformer.layers.0.self_attn.q_proj.lora_B
base_model.model.model.vision_model.transformer.layers.0.self_attn.q_proj.lora_B.default
base_model.model.model.vision_model.transformer.layers.0.self_attn.q_proj.lora_embedding_A
base_model.model.model.vision_model.transformer.layers.0.self_attn.q_proj.lora_embedding_B
base_model.model.model.vision_model.transformer.layers.0.self_attn.q_proj.lora_magnitude_vector
base_model.model.model.vision_model.transformer.layers.0.self_attn.k_proj.lora_dropout
base_model.model.model.vision_model.transformer.layers.0.self_attn.k_proj.lora_dropout.default
base_model.model.m

In [14]:
import random

random.seed(42)
shuffled = reports.copy()
random.shuffle(shuffled)

split_idx = int(len(shuffled) * 0.9)
train_reports = shuffled[:split_idx]
val_reports = shuffled[split_idx:]

print(f"Training examples:   {len(train_reports)}")
print(f"Validation examples: {len(val_reports)}")

train_dataset = DermatologyDataset(train_reports, processor)
val_dataset = DermatologyDataset(val_reports, processor)

print(f"\nDataset objects created")

Training examples:   2138
Validation examples: 238

Dataset objects created


In [16]:
training_args = SFTConfig(
    output_dir=f'{PROJECT_DIR}/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

print("Training config created")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

Training config created
Effective batch size: 8


In [19]:
from trl import SFTTrainer

train_dataset = DermatologyDataset(train_reports, processor)
val_dataset = DermatologyDataset(val_reports, processor)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=None,
)

print("Trainer created successfully")

Trainer created successfully


In [20]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'bos_token_id': 128000, 'pad_token_id': 128004}.
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:202: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(lbl) for lbl in labels]


Step,Training Loss,Validation Loss
100,0.937628,0.944830
200,0.860475,0.863853
300,0.783599,0.830686
400,0.791031,0.806648
500,0.741702,0.787301
600,0.722533,0.780710
700,0.708147,0.771790
800,0.711159,0.768168


/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:202: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(lbl) for lbl in labels]
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-packag

TrainOutput(global_step=804, training_loss=0.8323993066057044, metrics={'train_runtime': 4455.0288, 'train_samples_per_second': 1.44, 'train_steps_per_second': 0.18, 'total_flos': 1.31325308039573e+17, 'train_loss': 0.8323993066057044})

In [22]:
SAVE_PATH = f'{PROJECT_DIR}/checkpoints/final_model'
model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

Model saved to /content/drive/MyDrive/derma_project/checkpoints/final_model
